**Data Cleanup**

In [ ]:
"""
Data we need
- set of exams to schedule
- list of timeslots
- list of rooms available, by timeslot
- set of unique student schedules
- number of students with each unique schedule
- where classes during the semester
- set of classes that need to be split across rooms

Hard Constraints:
- room capacities
- all exams scheduled


Factors to consider:
1. student overlaps
2. 3 exams in 48 hours
3. back-to-back exams
4. in-person exams finish as early as possible
5. accomodate room requests
6. correct room type

If we have time:
- minimize exams on sunday
- fairness across majors

"""

In [ ]:
import pandas as pd
from datetime import datetime

# Load the CSV files
class_info = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Class_Info2023.csv')
final_exams = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Final_Exams2023.csv')
schedule = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Schedule2023.csv')
student_reg = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/StudentRegistration2023.csv')

# Process exams (E): list of CRNs from final_exams
E = final_exams['CRN'].tolist()
E = list(set(E))

# Class sizes: number of students per exam (CRN)
class_sizes = student_reg.groupby('CRN').size().to_dict()

# Timeslots (T): unique (EXAM_DATE, EXAM_TIME) from final_exams, sorted chronologically
timeslots = final_exams[['EXAM_DATE', 'EXAM_TIME']].drop_duplicates()
T = [(row['EXAM_DATE'], row['EXAM_TIME']) for _, row in timeslots.iterrows()]
T = sorted(T, key=lambda t: (datetime.strptime(t[0], '%m/%d/%Y'), t[1]))

# Rooms (R): unique rooms from class_info where AVAILABLE_TO_SCHEDULE == 'Y'
available_rooms = class_info[class_info['AVAILABLE_TO_SCHEDULE'] == 'Y']
rooms = available_rooms[['BUILDING_CODE', 'ROOM_NUMBER']].drop_duplicates()
R = [(row['BUILDING_CODE'], row['ROOM_NUMBER']) for _, row in rooms.iterrows()]

# Include dummy room pairs for oversized exams
R = R + [('dummy', '1'), ('dummy', '2'), ('dummy', '3')]

# Capacities: dict room -> ROOM_CAPACITY
capacities = {(row['BUILDING_CODE'], row['ROOM_NUMBER']): row['ROOM_CAPACITY'] for _, row in available_rooms.iterrows()}
capacities[('dummy', '1')] = capacities[('HRZ','AMP')] + capacities[('KCK', '100')]
capacities[('dummy', '2')] = capacities[('DCH', '1055')] + capacities[('HRG', '100')]
capacities[('dummy', '3')] = capacities[('HRZ', '210')] + capacities[('HRZ', '212')]

# Student schedules (S): dict index -> set of CRNs
student_schedules = student_reg.groupby('PERSON_IDENTIFIER')['CRN'].apply(set).to_dict()

# Unique schedules and counts (S and N)
from collections import defaultdict
schedule_counts = defaultdict(int)
schedule_to_index = {}
index = 0
for sched in student_schedules.values():
    sched_tuple = frozenset(sched)
    if sched_tuple not in schedule_to_index:
        schedule_to_index[sched_tuple] = index
        index += 1
    schedule_counts[schedule_to_index[sched_tuple]] += 1

S = {idx: set(sched) for sched, idx in schedule_to_index.items()}
N = schedule_counts

print(f"Number of exams:             {len(E)}")
print(f"Number of timeslots:         {len(T)}")
print(f"Number of rooms:             {len(R)}")
print(f"Number of unique schedules:  {len(S)}")
print(f"Timeslots (chronological):   {T}")


In [ ]:
import gurobipy as gp
from gurobipy import GRB
from collections import defaultdict

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

def schedule_model(E, T, R, S, N, capacities, class_sizes):
    """
    Inputs:
    E - list of CRNs (exams to schedule)
    T - ordered list of (EXAM_DATE, EXAM_TIME) timeslot tuples, sorted chronologically
    R - list of (BUILDING_CODE, ROOM_NUMBER) room tuples
    S - dict: schedule_idx -> set of CRNs
    N - dict: schedule_idx -> number of students with that schedule
    capacities - dict: (BUILDING_CODE, ROOM_NUMBER) -> capacity
    class_sizes - dict: CRN -> number of enrolled students
    """
    m = gp.Model(env=env)

    E_set = set(E)
    sched_keys = list(S.keys())

    # CRNs per schedule that have a final exam
    exam_crns_by_sched = {
        idx: [crn for crn in crns if crn in E_set]
        for idx, crns in S.items()
    }

    # Compute student overlap counts efficiently:
    # iterate over schedules rather than all O(|E|^2) pairs
    overlap = defaultdict(int)
    for idx, crns in S.items():
        exam_crns = [crn for crn in crns if crn in E_set]
        for i in range(len(exam_crns)):
            for j in range(i + 1, len(exam_crns)):
                pair = (min(exam_crns[i], exam_crns[j]), max(exam_crns[i], exam_crns[j]))
                overlap[pair] += N[idx]

    # Only build z variables for pairs that actually share students (3.7% of all pairs)
    E_pairs = list(overlap.keys())
    print(f"Nonzero-overlap pairs: {len(E_pairs):,} (vs {len(E)*(len(E)-1)//2:,} total)")

    # DECISION VARIABLES
    x = m.addVars(E, T, R, vtype=GRB.BINARY, name="x")
    y = m.addVars(E, T, vtype=GRB.BINARY, name="y")
    z = m.addVars(E_pairs, T, vtype=GRB.CONTINUOUS, name="z")

    # Blocking constraints for dummy room pairs:
    # when any exam uses a dummy room at time t, both constituent physical rooms
    # must be free (at most one "event" can claim each physical room per slot)
    m.addConstrs(
        (gp.quicksum(x[e, t, ('HRZ', 'AMP')]   for e in E) +
         gp.quicksum(x[e, t, ('dummy', '1')]    for e in E) <= 1 for t in T),
        name="block_dummy1_hrz")
    m.addConstrs(
        (gp.quicksum(x[e, t, ('KCK', '100')]   for e in E) +
         gp.quicksum(x[e, t, ('dummy', '1')]    for e in E) <= 1 for t in T),
        name="block_dummy1_kck")
    m.addConstrs(
        (gp.quicksum(x[e, t, ('DCH', '1055')]  for e in E) +
         gp.quicksum(x[e, t, ('dummy', '2')]    for e in E) <= 1 for t in T),
        name="block_dummy2_dch")
    m.addConstrs(
        (gp.quicksum(x[e, t, ('HRG', '100')]   for e in E) +
         gp.quicksum(x[e, t, ('dummy', '2')]    for e in E) <= 1 for t in T),
        name="block_dummy2_hrg")
    m.addConstrs(
        (gp.quicksum(x[e, t, ('HRZ', '210')]   for e in E) +
         gp.quicksum(x[e, t, ('dummy', '3')]    for e in E) <= 1 for t in T),
        name="block_dummy3_hrz210")
    m.addConstrs(
        (gp.quicksum(x[e, t, ('HRZ', '212')]   for e in E) +
         gp.quicksum(x[e, t, ('dummy', '3')]    for e in E) <= 1 for t in T),
        name="block_dummy3_hrz212")

    # Timeslot weights: index 1...|T|, so earlier slots cost less (objective minimizes)
    w = {t: i + 1 for i, t in enumerate(T)}

    # TRACKING 3 EXAMS IN 48 HOURS
    # T has 3 slots/day; a 6-slot window covers exactly 48 hours
    window_starts = T[:-5]
    v = m.addVars(window_starts, sched_keys, vtype=GRB.INTEGER, name="v")

    for t_idx, t_start in enumerate(window_starts):
        window = T[t_idx:t_idx + 6]
        for sched_idx in sched_keys:
            m.addConstr(
                v[t_start, sched_idx] == gp.quicksum(
                    x[crn, t, r]
                    for crn in exam_crns_by_sched[sched_idx]
                    for t in window
                    for r in R
                ),
                name=f"v_def[{t_start},{sched_idx}]"
            )

    u = m.addVars(window_starts, sched_keys, vtype=GRB.BINARY, name="u")
    for t_start in window_starts:
        for sched_idx in sched_keys:
            m.addGenConstrIndicator(u[t_start, sched_idx], True,  v[t_start, sched_idx] >= 3)
            m.addGenConstrIndicator(u[t_start, sched_idx], False, v[t_start, sched_idx] <= 2)

    # TRACKING BACK-TO-BACK EXAMS
    # p[t, sched_idx] = 1 iff schedule sched_idx has at least one exam at time t.
    # Using a binary indicator avoids infeasibility when a student has 2+ exams
    # in the same slot (which the overlap penalty allows during branch-and-bound).
    p = m.addVars(T, sched_keys, vtype=GRB.BINARY, name="p")
    for t in T:
        for sched_idx in sched_keys:
            crns = exam_crns_by_sched[sched_idx]
            if crns:
                m.addConstrs((p[t, sched_idx] >= y[crn, t] for crn in crns))
                m.addConstr(p[t, sched_idx] <= gp.quicksum(y[crn, t] for crn in crns))
            else:
                m.addConstr(p[t, sched_idx] == 0)

    consec_starts = T[:-1]
    d = m.addVars(consec_starts, sched_keys, vtype=GRB.BINARY, name="d")
    for t_idx, t in enumerate(consec_starts):
        t_next = T[t_idx + 1]
        for sched_idx in sched_keys:
            m.addConstr(d[t, sched_idx] <= p[t,      sched_idx])
            m.addConstr(d[t, sched_idx] <= p[t_next, sched_idx])
            m.addConstr(d[t, sched_idx] >= p[t, sched_idx] + p[t_next, sched_idx] - 1)

    # OBJECTIVE
    lamb = 1
    mu   = 10  # penalty for 3 exams in 48 hours
    nu   = 5   # penalty for back-to-back

    m.setObjective(
        gp.quicksum(w[t] * x[crn, t, r] for crn in E for t in T for r in R) +
        lamb * gp.quicksum(overlap[(c1, c2)] * z[c1, c2, t] for c1, c2 in E_pairs for t in T) +
        mu   * gp.quicksum(u[t_start, idx]  for t_start in window_starts for idx in sched_keys) +
        nu   * gp.quicksum(d[t, idx]        for t in consec_starts        for idx in sched_keys),
        GRB.MINIMIZE
    )

    # HARD CONSTRAINTS

    # Each exam scheduled exactly once
    m.addConstrs(
        (gp.quicksum(x[crn, t, r] for t in T for r in R) == 1 for crn in E),
        name="assign_once"
    )

    # Room capacities
    m.addConstrs(
        (gp.quicksum(class_sizes[crn] * x[crn, t, r] for crn in E) <= capacities[r]
         for t in T for r in R),
        name="capacity"
    )

    # y[crn, t] = 1 iff crn is scheduled at t
    m.addConstrs(
        (y[crn, t] == gp.quicksum(x[crn, t, r] for r in R) for crn in E for t in T),
        name="y_def"
    )

    # Linearize z[c1, c2, t] = y[c1, t] * y[c2, t]
    m.addConstrs((z[c1, c2, t] <= y[c1, t]                 for c1, c2 in E_pairs for t in T))
    m.addConstrs((z[c1, c2, t] <= y[c2, t]                 for c1, c2 in E_pairs for t in T))
    m.addConstrs((z[c1, c2, t] >= y[c1, t] + y[c2, t] - 1 for c1, c2 in E_pairs for t in T))

    m.optimize()

    if m.Status in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
        print(f"Objective value: {m.ObjVal:.2f}")
        schedule_result = {}
        for crn in E:
            for t in T:
                for r in R:
                    if x[crn, t, r].X > 0.5:
                        schedule_result[crn] = {'timeslot': t, 'room': r}
        return schedule_result
    else:
        print(f"No optimal solution found. Status: {m.Status}")
        return None


In [ ]:
import json as _json

result = schedule_model(E, T, R, S, N, capacities, class_sizes)

if result is not None:
    # Convert tuple keys to serializable strings
    output = {
        str(crn): {
            'date': v['timeslot'][0],
            'time': v['timeslot'][1],
            'building': v['room'][0],
            'room': v['room'][1]
        }
        for crn, v in result.items()
    }
    out_path = '/Users/benfreidinger/Desktop/decisionlab26/exam_schedule_optimized.json'
    with open(out_path, 'w') as f:
        _json.dump(output, f, indent=2)
    print(f'Schedule written to {out_path}')
